# ch01 数据清洗

本章节对 A/B 测试原始数据进行质量检查与清洗。

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# 添加项目根目录到路径
PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Config
from src.utils.data_loader import DataLoader
from src.utils.metrics import Metrics
from src.utils.output_manager import OutputManager

print("模块加载成功")

## Step 1: 加载原始数据

In [ ]:
config = Config()
loader = DataLoader(config)
metrics = Metrics()
output = OutputManager(config)

df = loader.load_raw()
print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
print(f"数据类型:\n{df.dtypes}")
df.head()

## Step 2: 缺失值检查

In [ ]:
missing_summary = metrics.null_summary(df)
print("缺失值统计:")
print(missing_summary)
output.save_table(missing_summary, "missing_values_summary", chapter_prefix="ch01")

## Step 3: 重复值检查

In [ ]:
dup_summary = metrics.duplicate_summary(df)
print("重复值统计:")
print(dup_summary)
output.save_json(dup_summary, "duplicate_summary", chapter_prefix="ch01")

## Step 4: 数据类型验证与转换

In [ ]:
# 转换 timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"timestamp 类型: {df['timestamp'].dtype}")

# 验证 click 取值
print(f"click 唯一值: {df['click'].unique()}")

# 验证 group 取值
print(f"group 唯一值: {df['group'].unique()}")

# 验证 user_id 唯一性
print(f"user_id 唯一数量: {df['user_id'].nunique()}")

## Step 5: 描述性统计

In [ ]:
# 分组统计
group_stats = df.groupby('group').agg({
    'user_id': 'count',
    'click': ['sum', 'mean']
}).round(4)
group_stats.columns = ['用户数', '点击数', '点击率']
print("分组点击统计:")
print(group_stats)
output.save_table(group_stats.reset_index(), "group_click_stats", chapter_prefix="ch01")

## Step 6: 保存清洗后数据

In [ ]:
output.save_data(df, "cleaned_data.csv")
print("清洗后数据已保存到 data/processed/cleaned_data.csv")